In [ ]:
const int PIN_LIGHT = 8;
const int PIN_DOOR  = 9;
const int PIN_AC    = 10;
const int PIN_FAN   = 11;
const int PIN_LCD   = 12;

const String TEACHER_ID = "22-46342-1";
const String STUDENT1_ID = "22-47180-1";
const String STUDENT2_ID = "22-47892-2";

String currentUser = "UNKNOWN";
bool isTeacher = false;
bool isStudent = false;
bool isAuthorized = false;

void setup() {
  Serial.begin(9600);

  pinMode(PIN_LIGHT, OUTPUT);
  pinMode(PIN_DOOR, OUTPUT);
  pinMode(PIN_AC, OUTPUT);
  pinMode(PIN_FAN, OUTPUT);
  pinMode(PIN_LCD, OUTPUT);

  digitalWrite(PIN_LIGHT, HIGH);
  digitalWrite(PIN_DOOR, HIGH);
  digitalWrite(PIN_AC, HIGH);
  digitalWrite(PIN_FAN, HIGH);
  digitalWrite(PIN_LCD, HIGH);

  Serial.println("[ARDUINO] System Ready");
  Serial.println("[ARDUINO] Waiting for commands...");

  for(int i = 0; i < 3; i++) {
    digitalWrite(PIN_LIGHT, LOW);
    delay(150);
    digitalWrite(PIN_LIGHT, HIGH);
    delay(150);
  }

  Serial.println("[ARDUINO] Startup blink complete");
}

void loop() {
  if (Serial.available() > 0) {
    String command = Serial.readStringUntil('\n');
    command.trim();

    if (command.length() == 0) return;

    Serial.print("[ARDUINO] Received: '");
    Serial.print(command);
    Serial.println("'");

    if (command == TEACHER_ID) {
      currentUser = TEACHER_ID;
      isTeacher = true;
      isStudent = false;
      isAuthorized = true;
      Serial.println("[ARDUINO] ✓ TEACHER authorized - Full access");
    }
    else if (command == STUDENT1_ID || command == STUDENT2_ID) {
      currentUser = command;
      isTeacher = false;
      isStudent = true;
      isAuthorized = true;
      Serial.print("[ARDUINO] ✓ STUDENT authorized - ID: ");
      Serial.println(command);
    }
    else if (command == "UNKNOWN") {
      currentUser = "UNKNOWN";
      isTeacher = false;
      isStudent = false;
      isAuthorized = false;
      Serial.println("[ARDUINO] ✗ UNAUTHORIZED - Access denied");
      turnAllOff();
    }

    else if (command.indexOf(';') != -1) {
      processMultipleCommands(command);
    }
    else {
      processSingleCommand(command);
    }
  }
}

void processSingleCommand(String cmd) {
  if (!isAuthorized) {
    Serial.println("[ARDUINO] ✗ Command denied - Not authorized");
    return;
  }

  if (cmd == "L1") {
    digitalWrite(PIN_LIGHT, LOW);
    Serial.println("[ARDUINO] ✓ Light ON (Pin 8 = LOW)");
  }
  else if (cmd == "L0") {
    digitalWrite(PIN_LIGHT, HIGH);
    Serial.println("[ARDUINO] ✓ Light OFF (Pin 8 = HIGH)");
  }

  else if (cmd == "F1") {
    digitalWrite(PIN_FAN, LOW);
    Serial.println("[ARDUINO] ✓ Fan ON (Pin 11 = LOW)");
  }
  else if (cmd == "F0") {
    digitalWrite(PIN_FAN, HIGH);
    Serial.println("[ARDUINO] ✓ Fan OFF (Pin 11 = HIGH)");
  }

  else if (cmd == "A1") {
    digitalWrite(PIN_AC, LOW);
    Serial.println("[ARDUINO] ✓ AC ON (Pin 10 = LOW)");
  }
  else if (cmd == "A0") {
    digitalWrite(PIN_AC, HIGH);
    Serial.println("[ARDUINO] ✓ AC OFF (Pin 10 = HIGH)");
  }

  else if (cmd == "D1") {
    if (isTeacher) {
      digitalWrite(PIN_DOOR, LOW);
      Serial.println("[ARDUINO] ✓ Door UNLOCKED (Pin 9 = LOW) - Teacher");
    } else {
      Serial.println("[ARDUINO] ✗ Door access DENIED - Student cannot unlock");
    }
  }
  else if (cmd == "D0") {
    if (isTeacher) {
      digitalWrite(PIN_DOOR, HIGH);
      Serial.println("[ARDUINO] ✓ Door LOCKED (Pin 9 = HIGH) - Teacher");
    } else {
      Serial.println("[ARDUINO] ✗ Door access DENIED - Student cannot lock");
    }
  }

  else if (cmd == "LD1") {
    if (isTeacher) {
      digitalWrite(PIN_LCD, LOW);
      Serial.println("[ARDUINO] ✓ LCD ON (Pin 12 = LOW) - Teacher");
    } else {
      Serial.println("[ARDUINO] ✗ LCD access DENIED - Student cannot control");
    }
  }
  else if (cmd == "LD0") {
    if (isTeacher) {
      digitalWrite(PIN_LCD, HIGH);
      Serial.println("[ARDUINO] ✓ LCD OFF (Pin 12 = HIGH) - Teacher");
    } else {
      Serial.println("[ARDUINO] ✗ LCD access DENIED - Student cannot control");
    }
  }

  else {
    Serial.print("[ARDUINO] ✗ Unknown command: '");
    Serial.print(cmd);
    Serial.println("'");
  }
}

void processMultipleCommands(String multiCmd) {
  if (!isAuthorized) {
    Serial.println("[ARDUINO] ✗ Multi-command denied - Not authorized");
    return;
  }

  Serial.println("[ARDUINO] Processing multi-command...");

  int startPos = 0;
  int endPos = 0;

  while (endPos != -1) {
    endPos = multiCmd.indexOf(';', startPos);
    String subCmd;

    if (endPos == -1) {
      subCmd = multiCmd.substring(startPos);
    } else {
      subCmd = multiCmd.substring(startPos, endPos);
    }

    subCmd.trim();

    if (subCmd.length() > 0) {
      if (subCmd == "L1") {
        digitalWrite(PIN_LIGHT, LOW);
        Serial.println("[ARDUINO]   ✓ Light ON");
      }
      else if (subCmd == "L0") {
        digitalWrite(PIN_LIGHT, HIGH);
        Serial.println("[ARDUINO]   ✓ Light OFF");
      }

      else if (subCmd == "F1") {
        digitalWrite(PIN_FAN, LOW);
        Serial.println("[ARDUINO]   ✓ Fan ON");
      }
      else if (subCmd == "F0") {
        digitalWrite(PIN_FAN, HIGH);
        Serial.println("[ARDUINO]   ✓ Fan OFF");
      }

      else if (subCmd == "A1") {
        digitalWrite(PIN_AC, LOW);
        Serial.println("[ARDUINO]   ✓ AC ON");
      }
      else if (subCmd == "A0") {
        digitalWrite(PIN_AC, HIGH);
        Serial.println("[ARDUINO]   ✓ AC OFF");
      }

      else if (subCmd == "D1") {
        if (isTeacher) {
          digitalWrite(PIN_DOOR, LOW);
          Serial.println("[ARDUINO]   ✓ Door UNLOCKED");
        } else {
          Serial.println("[ARDUINO]   ✗ Door DENIED (Student)");
        }
      }
      else if (subCmd == "D0") {
        if (isTeacher) {
          digitalWrite(PIN_DOOR, HIGH);
          Serial.println("[ARDUINO]   ✓ Door LOCKED");
        }
      }

      else if (subCmd == "LD1") {
        if (isTeacher) {
          digitalWrite(PIN_LCD, LOW);
          Serial.println("[ARDUINO]   ✓ LCD ON");
        } else {
          Serial.println("[ARDUINO]   ✗ LCD DENIED (Student)");
        }
      }
      else if (subCmd == "LD0") {
        if (isTeacher) {
          digitalWrite(PIN_LCD, HIGH);
          Serial.println("[ARDUINO]   ✓ LCD OFF");
        }
      }
    }

    startPos = endPos + 1;
  }

  Serial.println("[ARDUINO] Multi-command complete");
}

void turnAllOff() {
  digitalWrite(PIN_LIGHT, HIGH);
  digitalWrite(PIN_FAN, HIGH);
  digitalWrite(PIN_AC, HIGH);
  digitalWrite(PIN_DOOR, HIGH);
  digitalWrite(PIN_LCD, HIGH);
  Serial.println("[ARDUINO] ✓ All components OFF");
}